***PREPROCESSING — STEP 1: Caricamento dataset***

In [17]:
# Import librerie
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Carico il CSV in un DataFrame
df = pd.read_csv("TicketEstrazione050326.csv")

***PREPROCESSING — STEP 2: Normalizzazione campo area***

In [18]:
# Mapping completo chiave → valore dei campi Area numerici, basato sulla tabella zhc_dropdown_maia
AREA_MAPPING = {
    '1': 'Amministrazione',
    '2': 'Amm-Economato',
    '3': 'Utenti e Ospiti',
    '4': 'CSS',
    '5': 'Personale',
    '6': 'Hardware',
    '7': 'CCE',
    '8': 'CBA DR STP',
}

# Quanti ticket hanno area numerica?
area_numerica_mask = df['area'].astype(str).str.strip().isin(AREA_MAPPING.keys())
print(f"Ticket con area numerica da mappare: {area_numerica_mask.sum():,}")
print(df[area_numerica_mask]['area'].value_counts())

# Applico mapping
df['area'] = df['area'].apply(
    lambda x: AREA_MAPPING.get(str(x).strip(), x) if pd.notna(x) else x
)

print(f"\nDistribuzione area dopo mapping:")
print(df['area'].value_counts(dropna=False))

Ticket con area numerica da mappare: 808
area
6    797
1      4
5      3
4      3
3      1
Name: count, dtype: int64

Distribuzione area dopo mapping:
area
area_personale                     14020
ciclo_passivo                      10756
ciclo_attivo                        9018
area_sanitaria                      8461
rendicontazione_flussi              2827
protocollo_documentale_delibere     2508
NaN                                 1802
area_sistemistica                   1069
sistema381                           933
Hardware                             797
area_territoriale                    525
protocollo_delibere                  228
business_intelligence                118
Amministrazione                        4
Personale                              3
CSS                                    3
Utenti e Ospiti                        1
flussi_regionali                       1
Name: count, dtype: int64


***PREPROCESSING — STEP 3: Imputazione priorita_iniziale_cliente***

In [19]:
# Se trovo NaN in priorità iniziale = operatore NON ha modificato la priorità
# Il cliente aveva già scelto quella giusta
# priorita_iniziale_cliente == priorita_finale
# Imputo i NaN con priorita_finale

nan_prima = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN prima dell'imputazione: {nan_prima:,} ({nan_prima/len(df)*100:.1f}%)")

df['priorita_iniziale_cliente'] = df['priorita_iniziale_cliente'].fillna(df['priorita_finale'])

nan_dopo = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN dopo l'imputazione: {nan_dopo:,}")

# Distribuzione dopo imputazione
print(f"\nDistribuzione priorita_iniziale_cliente dopo imputazione:")
print(df['priorita_iniziale_cliente'].value_counts().sort_index())

NaN prima dell'imputazione: 46,239 (87.1%)
NaN dopo l'imputazione: 0

Distribuzione priorita_iniziale_cliente dopo imputazione:
priorita_iniziale_cliente
P1    12560
P2    14573
P3    24790
P4     1151
Name: count, dtype: int64


***PREPROCESSING — STEP 4: Pulizia testi***

In [20]:
# import librerie per pulizia testo
from bs4 import BeautifulSoup
import re

# Funzioni di pulizia testo

def pulisci_html(testo):
    """Pulizia base HTML usando BeautifulSoup."""
    # se il valore non è stringa o è vuoto/non significativo restituisco stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    # costruisco un DOM con BeautifulSoup
    soup = BeautifulSoup(testo, "html.parser")
    # ogni tag <br> viene sostituito con un newline
    for br in soup.find_all("br"):
        br.replace_with("\n")
    # per ogni <p> inserisco uno spazio dopo il tag e lo "sciolgo"
    # (unwrap rimuove il tag lasciando il contenuto)
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    # estraggo il testo finale e tolgo spazi iniziali/finali
    return soup.get_text().strip()


def pulisci_blocco_reperibilita(soup):
    """
    Rimuove il blocco 'Orario reperibilità + Telefono' 
    navigando il DOM.
    """
    # prendo tutti i <br> perchè da lì riconosco la struttura del blocco
    br_tags = soup.find_all("br")
    testo = soup.get_text()
    
    # se nel testo compare la stringa di reperibilità (con o senza accento)
    if "Orario reperibilità" in testo or "Orario reperibilita" in testo:
        # verifico che ci siano almeno due <br> (atteso formattazione del blocco)
        if len(br_tags) >= 2:
            second_br = br_tags[1]
            # estraggo tutti i nodi precedenti al secondo <br> (cioè l'intero blocco)
            for elem in list(second_br.previous_siblings):
                elem.extract()
            # estraggo anche il secondo <br> stesso
            second_br.extract()
    return soup


def pulisci_descrizione(testo):
    """
    Pulizia descrizione ticket:
    - Rimuove blocco reperibilità/telefono
    - Rimuove HTML
    """
    # guard clause: input non valido → stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # parse HTML
    soup = BeautifulSoup(testo, "html.parser")
    # elimino l'eventuale blocco di reperibilità
    soup = pulisci_blocco_reperibilita(soup)
    
    # normalizzo i tag <br> e <p> come in pulisci_html
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    
    testo_pulito = soup.get_text()
    # tolgo triple (o più) newline consecutivi -> lascio al massimo due
    testo_pulito = re.sub(r'\n{3,}', '\n\n', testo_pulito)
    return testo_pulito.strip()


def pulisci_conversazione(testo):
    """
    Pulizia conversazione completa:
    - Divide per separatore ---
    - Pulisce ogni messaggio singolarmente
    - Rimuove blocco reperibilità solo dal primo messaggio cliente
    - Ricostruisce la conversazione
    """
    # guard clause
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # i messaggi sono separati dal marcatore esplicito
    messaggi = testo.split('\n---\n')
    messaggi_puliti = []
    
    # itero su tutti i messaggi
    for i, messaggio in enumerate(messaggi):
        # cerco di separare l'autore dal corpo (formato 'AUTORE: testo')
        if ': ' in messaggio:
            autore, corpo = messaggio.split(': ', 1)
        else:
            autore, corpo = '', messaggio
        
        soup = BeautifulSoup(corpo, "html.parser")
        
        # solo se l'autore è CLIENTE rimuovo il blocco reperibilità
        if autore.upper() == 'CLIENTE':
            soup = pulisci_blocco_reperibilita(soup)
        
        # sostituisco <br> e <p> come prima
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for p in soup.find_all("p"):
            p.insert_after(" ")
            p.unwrap()
        
        corpo_pulito = soup.get_text().strip()
        
        # se il messaggio è vuoto dopo la pulizia lo salto
        if not corpo_pulito:
            continue
        
        # ricostruisco la stringa del messaggio con autore se presente
        if autore:
            if autore.upper() == 'CLIENTE':
                autore_norm = 'CLIENTE'
            else:
                autore_norm = 'OPERATORE'
            messaggi_puliti.append(f"{autore_norm}: {corpo_pulito}")
        else:
            messaggi_puliti.append(corpo_pulito)
    
    # riunisco la conversazione con lo stesso separatore iniziale
    return '\n---\n'.join(messaggi_puliti)

# Applica le funzioni
df['oggetto_pulito']        = df['oggetto'].apply(pulisci_html)
df['descrizione_pulita']    = df['descrizione'].apply(pulisci_descrizione)
df['conversazione_pulita']  = df['conversazione'].apply(pulisci_conversazione)

In [21]:
# Analisi lunghezza descrizione
df['desc_n_char'] = df['descrizione_pulita'].str.len()
df['desc_n_parole'] = df['descrizione_pulita'].str.split().str.len()

print("=== STATISTICHE LUNGHEZZA DESCRIZIONE ===")
print(df[['desc_n_char', 'desc_n_parole']].describe().round(1))
print(f"\nDescrizioni vuote o quasi (< 5 parole): {(df['desc_n_parole'] < 5).sum():,}")
print(f"Descrizioni molto corte (< 10 parole):   {(df['desc_n_parole'] < 10).sum():,}")

=== STATISTICHE LUNGHEZZA DESCRIZIONE ===
       desc_n_char  desc_n_parole
count      53074.0        53074.0
mean         367.4           53.3
std          597.4           82.0
min            0.0            0.0
25%          187.0           26.0
50%          284.0           41.0
75%          426.0           63.0
max        62548.0         8673.0

Descrizioni vuote o quasi (< 5 parole): 484
Descrizioni molto corte (< 10 parole):   1,908


***Test***

In [22]:
mask_rep = df['conversazione'].str.contains('reperibilit', case=False, na=False)
print(f"Ticket con blocco reperibilità nella conversazione grezza: {mask_rep.sum():,}")

if mask_rep.sum() > 0:
    idx = df[mask_rep].index[4]
    print("\n=== CONVERSAZIONE GREZZA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione'][:6000])
    print("\n=== CONVERSAZIONE PULITA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione_pulita'][:6000])

Ticket con blocco reperibilità nella conversazione grezza: 26,644

=== CONVERSAZIONE GREZZA (primi 6000 chars) ===
CLIENTE: <strong>Orario reperibilità contatto: </strong><br /><strong>Telefono: </strong><br /><div class="ql-editor"><p>CIAO IVAN,</p><p><br /></p><p>Stiamo cambiando connettività nelle varie strutture adesso presso la Casa di Riposo Villa del Pensionato - Via San Francesco 3 - 47017 Rocca San Casciano (FC) l'indirizzo IP pubblico è il seguente: 79.62.249.228</p><p><br /></p><p> </p><p><br /></p><p>Si segnala l'urgenza</p><p><br /></p><p> </p><p><br /></p><p>Cordiali Saluti</p></div>
---
Mauro Zaltron: <p>Effettuata modifica IP</p><br />
<p>attendo riscontro</p>
---
CLIENTE: <div class="ql-editor"><p>LA STRUTTURA MI HA COMUNICATO CHE FUNZIONANO I PROGRAMMI 1.0 CHE SI UTILLIZZANO TRAMITE DESKTOP REMOTO</p></div>
---
Mauro Zaltron: <div class="mozaik-inner" style="font-family:Arial, Helvetica, sans-serif;font-size:14px;line-height:22.4px;color:#444444;background-color:rgba(

In [23]:
# Visualizzo l'url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[idx, 'url_ticket']}")


URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=10160380-f462-cbd5-287a-67584f55e30f


***PREPROCESSING — STEP 6: Costruzione testo_input (oggetto_pulito + descrizione_pulita)***

In [24]:

# STEP 6: Costruzione testo_input
# Concatenazione oggetto_pulito + spazio + descrizione_pulita
# fillna('') per gestire i NaN, strip() per rimuovere spazi iniziali/finali
df['testo_input'] = (df['oggetto_pulito'].fillna('') + ' ' + df['descrizione_pulita'].fillna('')).str.strip()

# Statistiche
testi_vuoti = (df['testo_input'] == '').sum()
lunghezza_media = df['testo_input'].str.split().str.len().mean()

print(f"Testi vuoti (stringa vuota dopo strip): {testi_vuoti:,}")
print(f"Lunghezza media in parole:              {lunghezza_media:.1f}")

print(f"\nEsempio (primo testo_input non vuoto — primi 400 caratteri):")
esempio = df[df['testo_input'] != '']['testo_input'].iloc[2]
print(esempio[:400])


Testi vuoti (stringa vuota dopo strip): 0
Lunghezza media in parole:              58.3

Esempio (primo testo_input non vuoto — primi 400 caratteri):
Bilancio di previsione Buongiorno,
sto cercando di accedere a "Bilancio di previsione", ma mi compare l'errore in allegato.
Come devo procedere?
Grazie
Laura


***PREPROCESSING — STEP 7: Features derivate (has_caps, has_urgenza)***

In [25]:
def calc_has_caps(testo):
    if not isinstance(testo, str) or not testo.strip():
        return False
    parole = testo.split()
    if not parole:
        return False
    maiuscole = sum(1 for p in parole if p.isupper() and len(p) > 1)
    return (maiuscole / len(parole)) > 0.20

# --- has_urgenza ---
# True se contiene almeno una keyword di urgenza forte
URGENZA_KW = ['urgente', 'immediato', 'il prima possibile', 'critico']

def calc_has_urgenza(testo):
    if not isinstance(testo, str):
        return False
    testo_lower = testo.lower()
    return any(kw in testo_lower for kw in URGENZA_KW)

# Applica le tre funzioni
df['has_caps']         = df['testo_input'].apply(calc_has_caps)
df['has_urgenza']      = df['testo_input'].apply(calc_has_urgenza)

# Verifico distribuzione e correlazione con priorità
for col in ['has_caps', 'has_urgenza']:
    n_true = df[col].sum()
    pct = n_true / len(df) * 100
    print(f"\n{col}: {n_true:,} ticket ({pct:.1f}%)")
    print(pd.crosstab(df[col], df['priorita_finale'], normalize='index').mul(100).round(1))


has_caps: 4,223 ticket (8.0%)
priorita_finale    P1    P2    P3   P4
has_caps                              
False            14.4  31.0  51.8  2.8
True             20.9  33.9  43.5  1.7

has_urgenza: 2,941 ticket (5.5%)
priorita_finale    P1    P2    P3   P4
has_urgenza                           
False            13.0  31.0  53.1  2.8
True             47.0  35.7  17.1  0.3


***Salvataggio Dataframe Pulito***

In [26]:
# Salvo il DataFrame pulito in un nuovo CSV
df.to_csv("TicketEstrazione050326_pulito.csv", index=False)

In [27]:
df['n_parole'] = df['testo_input'].str.split().str.len()

print("=== Distribuzione lunghezze ===")
bins = [0, 5, 10, 20, 50, 100, float('inf')]
labels = ['<5', '5-9', '10-19', '20-49', '50-99', '100+']
print(pd.cut(df['n_parole'], bins=bins, labels=labels).value_counts().sort_index())

print(f"\nTicket con < 5 parole: {(df['n_parole'] < 5).sum():,}")

# Esempi da guardare manualmente
print("\n=== ESEMPI TESTI CORTI (< 5 parole) ===")
corti = df[df['n_parole'] < 5][['oggetto_pulito', 'descrizione_pulita', 'testo_input', 'priorita_finale']].copy()

for i, row in corti.sample(20, random_state=42).iterrows():
    print(f"\n--- priorità: {row['priorita_finale']} ---")
    print(f"oggetto:     {row['oggetto_pulito']}")
    print(f"descrizione: {row['descrizione_pulita']}")
    print(f"testo_input: {row['testo_input']}")

=== Distribuzione lunghezze ===
n_parole
<5         167
5-9        793
10-19     4192
20-49    25070
50-99    17428
100+      5424
Name: count, dtype: int64

Ticket con < 5 parole: 100

=== ESEMPI TESTI CORTI (< 5 parole) ===

--- priorità: P3 ---
oggetto:     installazione master
descrizione: Aggiornamento master
testo_input: installazione master Aggiornamento master

--- priorità: P2 ---
oggetto:     area contabile
descrizione: TS-730
testo_input: area contabile TS-730

--- priorità: P1 ---
oggetto:     estrapolazione dati sosia
descrizione: 
testo_input: estrapolazione dati sosia

--- priorità: P3 ---
oggetto:     aggiornamento
descrizione: 
testo_input: aggiornamento

--- priorità: P3 ---
oggetto:     FE1 2.0
descrizione: VERIFICHE
testo_input: FE1 2.0 VERIFICHE

--- priorità: P3 ---
oggetto:     Calcolo ore straordinarie
descrizione: 
testo_input: Calcolo ore straordinarie

--- priorità: P3 ---
oggetto:     aggiornamento
descrizione: 
testo_input: aggiornamento

--- priorità: P2 -

In [28]:
# PREPROCESSING — STEP 8: Pulizia testo_input e rimozione rumore

# --- 8a: Deduplicazione oggetto=descrizione nel testo_input ---
# Se oggetto_pulito e descrizione_pulita sono identici (case-insensitive),
# testo_input diventa solo oggetto_pulito (non concatenare due volte la stessa cosa)

mask_dup = (
    df['oggetto_pulito'].str.strip().str.lower() ==
    df['descrizione_pulita'].str.strip().str.lower()
)
df.loc[mask_dup, 'testo_input'] = df.loc[mask_dup, 'oggetto_pulito'].str.strip()

print(f"Ticket con oggetto == descrizione (deduplicati): {mask_dup.sum():,}")

# Ricalcola n_parole dopo la deduplication
df['n_parole'] = df['testo_input'].str.split().str.len()

# --- 8b: Flag e rimozione rumore puro ---
# Pattern generici che non danno informazione al classificatore

PATTERN_RUMORE = [
    r'^aggiornamento\.?$',
    r'^verifiche?\.?$',
    r'^prova allegati?\.?$',
    r'^per appuntamento\.?$',
    r'^richiesta precisazione\.?$',
    r'^appuntament[oi]\.?$',
    r'^buongiorno\.?$',
    r'^buongiorno,?\s*grazie\.?$',
    r'^vedi allegat[oi]\.?$',
    r'^si veda allegat[oi]\.?$',
    r'^come da allegat[oi]\.?$',
    r'^come da accordi\.?$',
    r'^vedi oggetto\.?$',
]

pattern_combinato = '|'.join(PATTERN_RUMORE)
mask_rumore = df['testo_input'].str.strip().str.lower().str.fullmatch(
    pattern_combinato.replace(r'\.?$', '').replace(r'^', ''),
    na=False
)

# Più robusto: usa str.contains con ^ e $ simulati su testo normalizzato
mask_rumore = df['testo_input'].str.strip().str.lower().str.match(
    f'({pattern_combinato})', na=False
)

n_rumore = mask_rumore.sum()
print(f"\nTicket rumore puro identificati: {n_rumore:,}")
print("\nEsempi ticket rumore:")
print(df[mask_rumore][['oggetto_pulito', 'descrizione_pulita', 'priorita_finale']].to_string())

# --- 8c: Drop ticket con < 3 parole uniche nel testo_input (soglia conservativa) ---
# Non droppiamo su n_parole grezzo perché "ERRORE CHIUSURA PIC" (3 parole) è classificabile
# Droppiamo solo se le parole UNICHE sono meno di 3

df['n_parole_uniche'] = df['testo_input'].apply(
    lambda t: len(set(t.lower().split())) if isinstance(t, str) else 0
)

mask_troppo_corto = df['n_parole_uniche'] < 2

print(f"\nTicket con < 2 parole uniche: {mask_troppo_corto.sum():,}")
print("\nEsempi:")
print(df[mask_troppo_corto][['testo_input', 'priorita_finale']].to_string())

# --- Drop finale ---
mask_drop = mask_rumore | mask_troppo_corto
print(f"\nTicket da rimuovere totali: {mask_drop.sum():,} ({mask_drop.mean()*100:.2f}%)")
print(f"Ticket rimanenti: {(~mask_drop).sum():,}")

df_clean = df[~mask_drop].copy()
df_clean = df_clean.drop(columns=['n_parole', 'n_parole_uniche'])

Ticket con oggetto == descrizione (deduplicati): 475

Ticket rumore puro identificati: 7

Esempi ticket rumore:
               oggetto_pulito      descrizione_pulita priorita_finale
5607   Richiesta precisazione  Richiesta precisazione              P3
12237           aggiornamento                                      P3
17366               VERIFICHE               VERIFICHE              P3
17958        per appuntamento        per appuntamento              P3
25868           aggiornamento                                      P3
49042          Prova allegati          Prova allegati              P3
49864           aggiornamento                                      P3

Ticket con < 2 parole uniche: 10

Esempi:
         testo_input priorita_finale
12237  aggiornamento              P3
17366      VERIFICHE              P3
18462     formazione              P3
23136          prova              P3
25868  aggiornamento              P3
26885           Test              P3
39416          PROVA      

In [29]:
df.columns

Index(['url_ticket', 'case_number', 'oggetto', 'descrizione',
       'priorita_finale', 'priorita_iniziale_cliente', 'area',
       'tipologia_intervento', 'articolo', 'modulo_sw', 'reparto',
       'data_creazione', 'data_risoluzione', 'conversazione', 'oggetto_pulito',
       'descrizione_pulita', 'conversazione_pulita', 'desc_n_char',
       'desc_n_parole', 'testo_input', 'has_caps', 'has_urgenza', 'n_parole',
       'n_parole_uniche'],
      dtype='str')